# Lab Exercise 3: Student Survey Simple Linear Regression
**Experiments**
1. CIA Percentage -> GPA
2. Attendance Percentage -> GPA


## Q1. Load Dataset and Export CSV

**Task covered:** Load the dataset using Pandas and use the collected survey data for analysis.

create `student_survey.csv`


In [1]:
# Q1: Import libraries, load the survey dataset, and create student_survey.csv
from pathlib import Path
import pickle
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

BASE_DIR = Path(".")
EXCEL_FILE = BASE_DIR / "Department Awareness Survey (Responses).xlsx"
CSV_FILE = BASE_DIR / "student_survey.csv"
PICKLE_FILE = BASE_DIR / "linear_regression_weights.pkl"

if CSV_FILE.exists():
    raw_survey = pd.read_csv(CSV_FILE)
elif EXCEL_FILE.exists():
    raw_survey = pd.read_excel(EXCEL_FILE)
    raw_survey.to_csv(CSV_FILE, index=False)
else:
    raise FileNotFoundError("Place student_survey.csv in this folder before running.")

print(f"Dataset loaded from: {CSV_FILE if CSV_FILE.exists() else EXCEL_FILE}")


Dataset loaded from: student_survey.csv


In [2]:
# Q2: Display the first 5 rows
# Display first 5 rows
raw_survey.head()


,Timestamp,Registration Number,Email,Job role that you are interested in\n,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Which is the highest paying company,Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester,Internships Interests
0,2026-06-22 08:49:31.362,2547123,jiyaelza.jabi@mca.christuniversity.in,Software Engineer,4,Option 1,9,Deolite,2,2,7,78,3.24,Option 1,Industry
1,2026-06-22 08:49:50.520,2547122,jinishaleema.rosario@mca.christuniversity.in,Data Engineer,"7,00,000",Option 1,"4,50,000",Deolite,4,3,"6,00,000",64.78,3.2,Option 1,Industry
2,2026-06-22 08:50:44.521,2547101,rajeev.chandar@mca.christuniversity.in,Software Engineer,6,17,8,DE Shaw,5,5,13,80,3.6,Option 1,Industry
3,2026-06-22 08:51:00.666,2547156,sounak.chakraborty@mca.christuniversity.in,Data Scientist,4.5,12,6.9,DE Shaw,3,4,7,70,3.2,92,Industry
4,2026-06-22 08:51:42.357,2547148,samar.subhash@mca.christuniversity.in,Software Engineer,8,12,10,EY,5,3,10,68,3.4,85,Industry


In [3]:
# Q3: Check dataset dimensions
# Check dataset dimensions
print("Rows, Columns:", raw_survey.shape)


Rows, Columns: (54, 15)


In [4]:
# Q4: Identify missing values
# Identify missing values
raw_survey.isnull().sum()


Timestamp                                                                                     0
Registration Number                                                                           0
Email                                                                                         0
Job role that you are interested in\n                                                         0
What is the minimum salary of students placed through campus (In LPA..respond as a number)    0
What is the maximum salary of students placed through campus (In LPA..respond as a number)    0
What is the  median salary of students placed through campus (In LPA..respond as a number)    0
Which is the highest paying company                                                           0
Rate your contribution towards extra curricular activities                                    0
Rate your technical competencies                                                              0
What are your package expectations (LPA)

In [5]:
# Q5: Check duplicate records before cleaning
# Remove duplicate records if present
print("Duplicate rows before cleaning:", raw_survey.duplicated().sum())


Duplicate rows before cleaning: 0


### Q5-Q7. Clean Data, Convert Datatypes, and Remove Duplicates

**Tasks covered:** Identify/handle null values, convert required columns into numerical datatype, remove duplicate records if present.

Before fitting a regression model, the columns used for calculation must contain actual numbers. Survey data often has small human-entry differences, so I cleaned the data instead of directly training on the raw values.

The form contains a few common survey-entry issues:

- some percentages are entered as proportions, such as `0.70` instead of `70`
- some invalid choice text such as `Option 1` appears in numeric columns
- a few GPA values are on a different scale from the observed 0-4 GPA scale

The cleaning rules below keep the raw CSV unchanged and create a cleaned analysis dataframe. I used this approach because changing the raw data directly would make it harder to prove what came from the form and what was changed during preprocessing.

Cleaning decisions used here:

- convert decimal percentages like `0.70` to `70`
- treat non-numeric option text as missing
- remove impossible or mixed-scale GPA values above `4`
- remove duplicate rows if any exist
- drop missing values separately inside each experiment so one missing attendance value does not unnecessarily remove a valid CIA-GPA record


In [6]:
# Q6-Q7: Convert selected columns to numeric values and clean invalid entries
COLUMN_MAP = {
    "your CIA % of last semester ": "CIA_Percentage",
    "your GPA of last semester": "GPA",
    "Your maximum attendance % till last semester": "Attendance_Percentage",
}


def to_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()
    if not text or "option" in text.lower():
        return np.nan

    text = text.replace(",", "").replace("%", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan


def normalize_percentage(value, minimum_plausible=30.0):
    number = to_number(value)
    if pd.isna(number):
        return np.nan
    if 0 < number <= 1:
        number *= 100
    if number < minimum_plausible or number > 100:
        return np.nan
    return float(number)


def normalize_gpa(value):
    number = to_number(value)
    if pd.isna(number) or number < 0 or number > 4:
        return np.nan
    return float(number)


student_data = raw_survey.copy()
student_data.columns = student_data.columns.str.strip()
student_data = student_data.rename(
    columns={original.strip(): clean for original, clean in COLUMN_MAP.items()}
)
student_data = student_data.drop_duplicates()

student_data["CIA_Percentage"] = student_data["CIA_Percentage"].apply(normalize_percentage)
student_data["Attendance_Percentage"] = student_data["Attendance_Percentage"].apply(normalize_percentage)
student_data["GPA"] = student_data["GPA"].apply(normalize_gpa)

required_columns = ["CIA_Percentage", "Attendance_Percentage", "GPA"]
student_data[required_columns].head()


,CIA_Percentage,Attendance_Percentage,GPA
0,78.00,NaN,3.24
1,64.78,NaN,3.20
2,80.00,NaN,3.60
3,70.00,92.0,3.20
4,68.00,85.0,3.40


In [7]:
# Q5: Check missing values after cleaning
# Missing values after numeric conversion and validation
student_data[required_columns].isnull().sum()


CIA_Percentage           3
Attendance_Percentage    6
GPA                      4
dtype: int64

In [8]:
# Q8: Generate statistical summary
# Statistical summary for selected numerical variables
student_data[required_columns].describe()


,CIA_Percentage,Attendance_Percentage,GPA
count,51.000000,48.000000,50.000000
mean,74.053137,93.121458,3.476800
std,6.577943,4.807110,0.269106
min,60.000000,85.000000,2.900000
25%,70.000000,89.000000,3.317500
50%,71.110000,93.500000,3.500000
75%,79.665000,97.992500,3.630000
max,89.000000,100.000000,4.000000


### Q8-Q9. Statistical Summary and Variable Selection

**Tasks covered:** Generate statistical summary and select dependent/independent variables.

- Dependent variable (Y): `GPA`
- Independent variable for Experiment 1 (X): `CIA_Percentage`
- Independent variable for Experiment 2 (X): `Attendance_Percentage`

GPA is selected as the dependent variable because the problem asks whether CIA percentage and attendance percentage can predict academic performance. CIA percentage and attendance percentage are used one at a time because this lab is about **simple** linear regression, where only one independent variable is used in each model.

Rows with missing values are removed separately for each experiment. I did this because CIA and attendance have different missing/invalid entries. If I removed all rows with any missing value at the beginning, I would lose some valid records unnecessarily.


## Q10-Q15. Regression Function for Scikit-learn and Manual OLS

**Tasks covered:** Split train/test data, train `LinearRegression`, obtain slope/intercept, predict values, manually compute OLS slope/intercept, and compare both methods.

The dataset is split into training and testing data so the model is fitted on one part and evaluated on unseen records. This is better than checking only on the same data used for training, because training performance alone can give an overly positive impression.

I used `random_state=42` so the same train-test split is produced each time the notebook runs. This makes the output reproducible for submission and viva.

Manual OLS formulas:

`slope = sum((x_i - x_mean) * (y_i - y_mean)) / sum((x_i - x_mean)^2)`

`intercept = y_mean - slope * x_mean`

Both methods are fitted on the same training split and compared on the same test split. This makes the comparison fair: any difference would come from the calculation method, not from using different data.


In [9]:
# Q10-Q15: Define scikit-learn regression and manual OLS computation
def ols_parameters(x_values, y_values):
    x_mean = np.mean(x_values)
    y_mean = np.mean(y_values)

    numerator = np.sum((x_values - x_mean) * (y_values - y_mean))
    denominator = np.sum((x_values - x_mean) ** 2)

    if denominator == 0:
        raise ValueError("OLS slope is undefined because all X values are identical.")

    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return float(slope), float(intercept)


def run_experiment(df, feature_col, target_col="GPA", test_size=0.2, random_state=42):
    data = df[[feature_col, target_col]].dropna().copy()

    X = data[[feature_col]].to_numpy()
    y = data[target_col].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    sklearn_slope = float(model.coef_[0])
    sklearn_intercept = float(model.intercept_)
    sklearn_predictions = model.predict(X_test)

    manual_slope, manual_intercept = ols_parameters(X_train.ravel(), y_train)
    manual_predictions = manual_slope * X_test.ravel() + manual_intercept

    comparison = pd.DataFrame(
        {
            feature_col: X_test.ravel(),
            "Actual_GPA": y_test,
            "Sklearn_Prediction": sklearn_predictions,
            "Manual_OLS_Prediction": manual_predictions,
            "Absolute_Difference": np.abs(sklearn_predictions - manual_predictions),
        }
    )

    return {
        "feature": feature_col,
        "target": target_col,
        "rows_used": len(data),
        "sklearn_slope": sklearn_slope,
        "sklearn_intercept": sklearn_intercept,
        "manual_slope": manual_slope,
        "manual_intercept": manual_intercept,
        "mse": mean_squared_error(y_test, sklearn_predictions),
        "r2_score": r2_score(y_test, sklearn_predictions),
        "comparison": comparison,
    }


## Q16. Experiment 1: CIA Percentage Predicting GPA

**Independent variable (X):** `CIA_Percentage`  
**Dependent variable (Y):** `GPA`


In [10]:
# Q16: Run Experiment 1, CIA Percentage predicting GPA
cia_result = run_experiment(student_data, "CIA_Percentage")

print("Rows used:", cia_result["rows_used"])
print("Scikit-learn slope:", cia_result["sklearn_slope"])
print("Scikit-learn intercept:", cia_result["sklearn_intercept"])
print("Manual OLS slope:", cia_result["manual_slope"])
print("Manual OLS intercept:", cia_result["manual_intercept"])
print("MSE:", cia_result["mse"])
print("R2 score:", cia_result["r2_score"])

cia_result["comparison"].round(6)


Rows used: 48
Scikit-learn slope: 0.0231431550590347
Scikit-learn intercept: 1.7794610843443546
Manual OLS slope: 0.023143155059034703
Manual OLS intercept: 1.7794610843443541
MSE: 0.041738676714581716
R2 score: -0.3968299827509689


,CIA_Percentage,Actual_GPA,Sklearn_Prediction,Manual_OLS_Prediction,Absolute_Difference
0,77.00,3.60,3.561484,3.561484,0.0
1,75.00,3.00,3.515198,3.515198,0.0
2,70.00,3.40,3.399482,3.399482,0.0
3,71.50,3.46,3.434197,3.434197,0.0
4,60.00,3.40,3.168050,3.168050,0.0
5,70.00,3.27,3.399482,3.399482,0.0
6,64.67,3.40,3.276129,3.276129,0.0
7,70.00,3.60,3.399482,3.399482,0.0
8,68.00,3.40,3.353196,3.353196,0.0
9,85.00,3.60,3.746629,3.746629,0.0


## Q17. Experiment 2: Attendance Percentage Predicting GPA

**Independent variable (X):** `Attendance_Percentage`  
**Dependent variable (Y):** `GPA`


In [11]:
# Q17: Run Experiment 2, Attendance Percentage predicting GPA
attendance_result = run_experiment(student_data, "Attendance_Percentage")

print("Rows used:", attendance_result["rows_used"])
print("Scikit-learn slope:", attendance_result["sklearn_slope"])
print("Scikit-learn intercept:", attendance_result["sklearn_intercept"])
print("Manual OLS slope:", attendance_result["manual_slope"])
print("Manual OLS intercept:", attendance_result["manual_intercept"])
print("MSE:", attendance_result["mse"])
print("R2 score:", attendance_result["r2_score"])

attendance_result["comparison"].round(6)


Rows used: 44
Scikit-learn slope: 0.019694991932589428
Scikit-learn intercept: 1.6388301121372382
Manual OLS slope: 0.019694991932589414
Manual OLS intercept: 1.6388301121372395
MSE: 0.06718634952848099
R2 score: 0.2522746954182411


,Attendance_Percentage,Actual_GPA,Sklearn_Prediction,Manual_OLS_Prediction,Absolute_Difference
0,88.00,3.50,3.371989,3.371989,0.0
1,96.00,3.86,3.529549,3.529549,0.0
2,94.00,3.60,3.490159,3.490159,0.0
3,97.99,3.77,3.568742,3.568742,0.0
4,89.00,3.30,3.391684,3.391684,0.0
5,99.34,3.74,3.595331,3.595331,0.0
6,98.00,3.89,3.568939,3.568939,0.0
7,91.00,2.90,3.431074,3.431074,0.0
8,95.00,3.40,3.509854,3.509854,0.0


## Q18. Compare Scikit-learn Predictions with Manual OLS Predictions

**Tasks covered:** Compare predictions from scikit-learn, predictions from manually computed OLS equation, and the difference between both outputs.

I compare the predictions because it proves that the mathematical OLS equation and scikit-learn's `LinearRegression` are solving the same simple linear regression problem.

The difference between scikit-learn predictions and manual OLS predictions should be zero or extremely close to zero. Very tiny differences can happen because computers store decimal values using floating-point representation, but in this notebook the differences are effectively `0.0`.


In [12]:
# Q18: Compare scikit-learn and manual OLS outputs
results = {
    "CIA_Percentage_to_GPA": cia_result,
    "Attendance_Percentage_to_GPA": attendance_result,
}

summary_rows = []
for experiment_name, result in results.items():
    max_difference = result["comparison"]["Absolute_Difference"].max()
    summary_rows.append(
        {
            "Experiment": experiment_name,
            "Rows_Used": result["rows_used"],
            "Sklearn_Slope": result["sklearn_slope"],
            "Sklearn_Intercept": result["sklearn_intercept"],
            "Manual_OLS_Slope": result["manual_slope"],
            "Manual_OLS_Intercept": result["manual_intercept"],
            "Max_Prediction_Difference": max_difference,
            "MSE": result["mse"],
            "R2_Score": result["r2_score"],
        }
    )

comparison_summary = pd.DataFrame(summary_rows)
comparison_summary.round(10)


,Experiment,Rows_Used,Sklearn_Slope,Sklearn_Intercept,Manual_OLS_Slope,Manual_OLS_Intercept,Max_Prediction_Difference,MSE,R2_Score
0,CIA_Percentage_to_GPA,48,0.023143,1.779461,0.023143,1.779461,0.0,0.041739,-0.396830
1,Attendance_Percentage_to_GPA,44,0.019695,1.638830,0.019695,1.638830,0.0,0.067186,0.252275


## Q19-Q21. Save, Load, and Reuse Parameters using Pickle

**Tasks covered:** Save slope/intercept, load the pickle file, and use loaded parameters for prediction.

The learned model parameters are slope and intercept. Since there are two regression experiments, both parameter sets are saved in the same pickle file.

Saving the parameters is useful because once the model is trained, we do not need to train it again just to make a prediction. We can load the slope and intercept from the pickle file and apply the regression equation:

`predicted GPA = slope * input value + intercept`


In [13]:
# Q19: Save learned slope/intercept parameters using pickle
weights = {
    experiment_name: {
        "feature": result["feature"],
        "target": result["target"],
        "slope": result["sklearn_slope"],
        "intercept": result["sklearn_intercept"],
        "manual_ols_slope": result["manual_slope"],
        "manual_ols_intercept": result["manual_intercept"],
    }
    for experiment_name, result in results.items()
}

with open(PICKLE_FILE, "wb") as file:
    pickle.dump(weights, file)

print(f"Saved parameters to {PICKLE_FILE}")
weights


Saved parameters to linear_regression_weights.pkl


{'CIA_Percentage_to_GPA': {'feature': 'CIA_Percentage', 'target': 'GPA', 'slope': 0.0231431550590347, 'intercept': 1.7794610843443546, 'manual_ols_slope': 0.023143155059034703, 'manual_ols_intercept': 1.7794610843443541}, 'Attendance_Percentage_to_GPA': {'feature': 'Attendance_Percentage', 'target': 'GPA', 'slope': 0.019694991932589428, 'intercept': 1.6388301121372382, 'manual_ols_slope': 0.019694991932589414, 'manual_ols_intercept': 1.6388301121372395}}

In [14]:
# Q20-Q21: Load pickle file and predict using loaded parameters
with open(PICKLE_FILE, "rb") as file:
    loaded_weights = pickle.load(file)


def predict_with_loaded_parameters(experiment_name, feature_value):
    params = loaded_weights[experiment_name]
    return params["slope"] * feature_value + params["intercept"]


print("Loaded parameters:")
print(loaded_weights)
print()
print("Prediction using loaded CIA model for CIA=75:", predict_with_loaded_parameters("CIA_Percentage_to_GPA", 75))
print("Prediction using loaded Attendance model for Attendance=85:", predict_with_loaded_parameters("Attendance_Percentage_to_GPA", 85))


Loaded parameters:
{'CIA_Percentage_to_GPA': {'feature': 'CIA_Percentage', 'target': 'GPA', 'slope': 0.0231431550590347, 'intercept': 1.7794610843443546, 'manual_ols_slope': 0.023143155059034703, 'manual_ols_intercept': 1.7794610843443541}, 'Attendance_Percentage_to_GPA': {'feature': 'Attendance_Percentage', 'target': 'GPA', 'slope': 0.019694991932589428, 'intercept': 1.6388301121372382, 'manual_ols_slope': 0.019694991932589414, 'manual_ols_intercept': 1.6388301121372395}}

Prediction using loaded CIA model for CIA=75: 3.5151977137719568
Prediction using loaded Attendance model for Attendance=85: 3.3129044264073393


## Q22. Final Observations and Inference

After cleaning the survey data, the regression models produced the following results:

| Experiment | Rows used | Slope | Intercept | Test MSE | Test R2 |
|---|---:|---:|---:|---:|---:|
| CIA Percentage -> GPA | 48 | 0.0231 | 1.7795 | 0.0417 | -0.3968 |
| Attendance Percentage -> GPA | 44 | 0.0197 | 1.6388 | 0.0672 | 0.2523 |

### Interpretation

For the CIA model, the slope is `0.0231`. This means that, according to the fitted line, a 1 percentage point increase in CIA is associated with an increase of about `0.023` GPA points. For example, increasing CIA by 10 percentage points would increase the predicted GPA by about `0.231` points.

For the attendance model, the slope is `0.0197`. This means that a 1 percentage point increase in attendance is associated with an increase of about `0.020` GPA points. For example, increasing attendance by 10 percentage points would increase the predicted GPA by about `0.197` points.

The manual OLS method and scikit-learn gave the same slope, intercept, and predictions. This happened because scikit-learn's `LinearRegression` also fits a least-squares line. The manual method helped verify the formula, while scikit-learn provided the library-based implementation.

### Which Predictor Worked Better?

In this train-test split, attendance had a better `R2` value (`0.2523`) than CIA (`-0.3968`). A positive R2 for attendance means the model explained some variation in GPA on the test data. The negative R2 for CIA means that, on the test split, the CIA regression line performed worse than simply predicting the average GPA.

So, for this particular cleaned dataset and split, attendance percentage was a slightly better predictor of GPA than CIA percentage. However, the result should not be treated as a strong conclusion because the dataset is small and based on self-reported survey values.

### Final Inference

Both CIA percentage and attendance percentage show a small positive relationship with GPA in the fitted lines. This means higher CIA or higher attendance generally leads to a higher predicted GPA*. Among the two, attendnce performed better on the test data, but the prediction strength is still moderate. More responses and more consistent data entry would be needed for a stronger and more reliable conclusion.

BUT,
Since the dataset is small, self-reported, and the model shows weak predictive power, there is not enough evidence to conclude a strong relationship between the variables and GPA. Therefore, we **fail to reject the null hypothesis** that there is no significant correlation.




## Q23. Sample Viva Questions and Answers

1. **What is Simple Linear Regression?** It is a supervised learning method that models the relationship between one independent variable and one dependent variable using a straight line.
2. **What is the role of slope and intercept?** The slope shows how much the predicted output changes when X increases by one unit. The intercept is the predicted output when X is zero.
3. **What is Ordinary Least Squares?** OLS is a method for finding the regression line that minimizes the sum of squared differences between actual and predicted values.
4. **Why do we square the errors in OLS?** Squaring makes all errors positive and penalizes larger errors more strongly.
5. **Difference between dependent and independent variable:** The independent variable is the input or predictor. The dependent variable is the output being predicted.
6. **Why should data be cleaned before training?** Missing, duplicate, invalid, or mixed-scale values can mislead the model and produce incorrect parameters.
7. **Why are slope and intercept called model parameters?** They are learned from the training data and define the fitted regression equation.
8. **Why do we save learned weights?** Saving parameters lets us reuse the trained model later without fitting it again.
9. **Can we accept the null hypothesis here?** It is better to say we fail to reject the null hypothesis, because the small self-reported dataset does not provide strong evidence of a significant correlation.
